# Progress snapshot — 2026 Q3 gas pipelines update

Weekly-ish view of how the update cycle is going, built from the live
"Pipelines (Gas/Oil/NGL) - main" sheet's `Researcher` + `LastUpdated` stamps,
optionally joined against the cycle update sheet's country assignments.

**Read-only** — this notebook never writes to any Google Sheet.

In [ ]:
%pip install -q -e ../../gem-tracker-constants

In [ ]:
import pandas as pd
import pygsheets

from gem_tracker_constants import GAS_FUEL_OPTIONS, PIPELINE_STATUS

pd.set_option('display.max_rows', 250)

## Config

In [ ]:
# live backend: "Pipelines (Gas/Oil/NGL) - main"
PIPELINES_SHEET_KEY = '1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek'
PIPELINES_TAB = 'Gas pipelines'
FUEL_OPTIONS = GAS_FUEL_OPTIONS

# rows with LastUpdated on/after this date count as touched this cycle
CYCLE_START = '2026-07-06'

# cycle update sheet (researcher/country assignments).
# TODO at spin-up: set to the "Q3 2026 gas pipelines - update sheet" key and
# share that sheet (viewer is enough) with the service account:
#   gem-analysis@gem-analysis.iam.gserviceaccount.com
# Model from last cycle (not shared with the service account, so not usable
# here): Q3 2025 sheet '1xK1qEj1uAsbyb1ekcFpjQzlHq_Nb2-fDuW9M7NCwiWI'
UPDATE_SHEET_KEY = None

# "in development" = the first two base statuses (proposed, construction);
# rollup defined in gem-tracker-constants statuses.yaml
IN_DEV_STATUSES = PIPELINE_STATUS[:2]

# flag researchers/countries with no stamps in this many days
STALE_DAYS = 14

## Pull the live sheet

In [ ]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
ss = gc.open_by_key(PIPELINES_SHEET_KEY)
df = ss.worksheet('title', PIPELINES_TAB).get_as_df(start='A3', include_tailing_empty=False)
df = df[df['Fuel'].isin(FUEL_OPTIONS)].copy()

df['LastUpdated_dt'] = pd.to_datetime(
    df['LastUpdated'].astype(str).str.strip(), errors='coerce', format='mixed')
df['updated_this_cycle'] = df['LastUpdated_dt'] >= pd.Timestamp(CYCLE_START)
df['in_dev'] = df['Status'].astype(str).str.strip().str.lower().isin(IN_DEV_STATUSES)

print(f'{len(df)} rows on "{PIPELINES_TAB}" with Fuel in {FUEL_OPTIONS}')
print(f'{int(df.updated_this_cycle.sum())} rows updated on/after {CYCLE_START}')

## Overall progress

In [ ]:
overall = pd.DataFrame({
    'rows': [int(df.in_dev.sum()), int((~df.in_dev).sum()), len(df)],
    'updated this cycle': [
        int((df.in_dev & df.updated_this_cycle).sum()),
        int((~df.in_dev & df.updated_this_cycle).sum()),
        int(df.updated_this_cycle.sum()),
    ],
}, index=['in development (priority 1)', 'other statuses (priority 2)', 'all'])
overall['% updated'] = (100 * overall['updated this cycle'] / overall['rows']).round(1)
overall

## By country

Rows are exploded on `CountriesOrAreas`, so multi-country pipelines count
toward every country they touch.

In [ ]:
by_country = (
    df.assign(country=df['CountriesOrAreas'].astype(str).str.split(','))
      .explode('country')
      .assign(country=lambda d: d['country'].str.strip())
)
by_country = by_country[by_country['country'] != '']
by_country['in_dev_updated'] = by_country['in_dev'] & by_country['updated_this_cycle']

country_progress = by_country.groupby('country').agg(
    rows=('ProjectID', 'size'),
    in_dev=('in_dev', 'sum'),
    in_dev_updated=('in_dev_updated', 'sum'),
    updated=('updated_this_cycle', 'sum'),
    last_activity=('LastUpdated_dt', 'max'),
)
country_progress['% in-dev updated'] = (
    100 * country_progress['in_dev_updated']
    / country_progress['in_dev'].where(country_progress['in_dev'] > 0)
).round(1)
country_progress['% all updated'] = (
    100 * country_progress['updated'] / country_progress['rows']).round(1)

country_progress.sort_values(
    ['% in-dev updated', 'in_dev'], ascending=[False, False])

## By researcher (rows stamped this cycle)

In [ ]:
touched = df[df['updated_this_cycle']].copy()
touched['Researcher'] = touched['Researcher'].astype(str).str.strip()

if touched.empty:
    print(f'No rows stamped on/after {CYCLE_START} yet.')
else:
    researcher_activity = touched.groupby('Researcher').agg(
        rows_touched=('ProjectID', 'size'),
        countries=('CountriesOrAreas', 'nunique'),
        first_activity=('LastUpdated_dt', 'min'),
        last_activity=('LastUpdated_dt', 'max'),
    ).sort_values('rows_touched', ascending=False)
    stale_cutoff = pd.Timestamp.today().normalize() - pd.Timedelta(days=STALE_DAYS)
    researcher_activity['stale?'] = researcher_activity['last_activity'] < stale_cutoff
    display(researcher_activity)

## Assignments vs activity (update sheet)

Joins the update sheet's researcher→country assignments against the live
sheet's activity, to spot assigned countries with no movement yet.

In [ ]:
if not UPDATE_SHEET_KEY:
    print('UPDATE_SHEET_KEY not set — skipping. Set it once the cycle update sheet exists.')
else:
    try:
        upd = (gc.open_by_key(UPDATE_SHEET_KEY)[0]
               .get_as_df(start='A2', include_tailing_empty=False))
    except Exception as e:
        raise RuntimeError(
            'Could not read the update sheet — is it shared (viewer) with '
            'gem-analysis@gem-analysis.iam.gserviceaccount.com?') from e
    upd.columns = [str(c).strip() for c in upd.columns]
    res_col = next(c for c in upd.columns if 'primary researcher' in c.lower())
    country_col = next(c for c in upd.columns if c.lower().startswith('country'))

    assignments = (upd[[res_col, country_col]]
                   .rename(columns={res_col: 'researcher', country_col: 'country'})
                   .astype(str).apply(lambda s: s.str.strip()))
    assignments = assignments[assignments['country'] != '']
    print(f'{len(assignments)} country assignments, '
          f'{assignments.researcher.replace("", pd.NA).nunique(dropna=True)} researchers')

    assigned = assignments.merge(
        country_progress.reset_index(), on='country', how='left')

    print('\nAssigned countries not matching any live-sheet country name:')
    unmatched = assigned[assigned['rows'].isna()]
    display(unmatched[['researcher', 'country']] if not unmatched.empty else 'none')

    print('\nAssigned countries with NO rows updated this cycle yet:')
    quiet = assigned[assigned['rows'].notna() & (assigned['updated'] == 0)]
    display(quiet[['researcher', 'country', 'rows', 'in_dev', 'last_activity']]
            .sort_values(['researcher', 'country'])
            .reset_index(drop=True))